# PREpiBind-ESMC-300M

Predicts whether a peptide binds an MHC class II allele, from the allele's two chains and the
epitope sequence. Run the four cells below in order.

This demo runs float16 throughout, and on Colab's T4 flash-attn is off, so its scores move in
the last decimals and do not reproduce the paper's numbers bit for bit.

Pick both chains from the dropdowns, or load a CSV with an `MHC` column that pairs the two
allele names as `beta_alpha` (e.g. `HLA-DRB1*01:01_HLA-DRA*01:01`). Every name must exist in
`demo/data/mhc_mapping_demo.csv`, the 116 alleles bundled with this repository.

In [ ]:
#@title 1. Setup { display-mode: "form" }
#@markdown Pick a model and run this cell. It clones the repository and downloads two float16
#@markdown checkpoints (~742 MB). Nothing is installed unless Colab's image is missing it:
#@markdown torch, huggingface_hub and the widget stack all ship with the runtime.
model_arm = "qualitative"  #@param ["qualitative", "ms", "ic50_500", "ic50_1000"]

import importlib, os, subprocess, sys
from IPython.display import clear_output


def _run(*cmd):
    """Run a command; on failure print its output into this cell and stop."""
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if p.returncode:
        print(p.stdout)
        raise RuntimeError(f"failed ({p.returncode}): {' '.join(cmd)}")


def _ensure(mod, pkg=None):
    """Import `mod`; pip-install it only if this runtime does not already have it."""
    try:
        return importlib.import_module(mod)
    except ImportError:
        _run(sys.executable, "-m", "pip", "install", "-q", pkg or mod)
        return importlib.import_module(mod)


REPO = "/content/PREpiBind"

# Shallow + blobless + sparse: a full checkout is ~114 MB, of which the demo needs ~20 MB.
# data/ and analysis/ are never fetched. Pin a tag here once v1.0.0 is cut; until then the clone
# tracks the tip of the default branch.
if not os.path.isdir(REPO):
    _run("git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
         "https://github.com/daylight-00/PREpiBind", REPO)
    # If the vendored ESM copy ever lands outside prepibind/, add its directory here.
    _run("git", "-C", REPO, "sparse-checkout", "set", "demo", "prepibind", "configs/predict")
os.chdir(os.path.join(REPO, "demo"))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# hf_transfer parallelises the download of the weights. Set the flag before huggingface_hub is
# imported -- that is when it is read -- and stand it back down if the package cannot be had.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
try:
    _ensure("hf_transfer")
except Exception:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("hf_transfer unavailable: falling back to the plain downloader.")
_ensure("huggingface_hub")
from huggingface_hub import hf_hub_download

ARMS = {  # arm -> (predict config, float16 checkpoint on daylight-00/prepibind-demo)
    "qualitative": ("config_demo.py",      "prepibind_qualitative_s100_f0_fp16.pt"),
    "ms":          ("config_ms.py",        "prepibind_ms_s128_f3_fp16.pt"),
    "ic50_500":    ("config_ic50_500.py",  "prepibind_ic50_500_s128_f2_fp16.pt"),
    "ic50_1000":   ("config_ic50_1000.py", "prepibind_ic50_1000_s42_f1_fp16.pt"),
}
config_name, chkp_name = ARMS[model_arm]
config_path = os.path.join(REPO, "configs", "predict", config_name)
MODELS = os.path.join(REPO, "models")

# Use the paths hf_hub_download returns; do not assume where local_dir put them.
esm_chkp_path = hf_hub_download("daylight-00/esmc-300m-2024-12", "esmc_300m_2024_12_v0_fp16.pth",
                                local_dir=MODELS)
chkp_path = hf_hub_download("daylight-00/prepibind-demo", chkp_name, local_dir=MODELS)

# Nothing needs installing for this import to work: prepibind carries its own copy of ESMC.
# If it raises, the checkout is incomplete -- fix it here, not at prediction time.
from prepibind.inference import main as run_inference, load_config

clear_output()
print(f"ready: {model_arm}  ({chkp_name})")

In [ ]:
#@title 2. Prepare Dataset
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd
import os

# The demo set: the 116 alleles the four dataset arms use, float16, already cut to the
# peptide-binding window. Built by demo/build_demo_assets.py; it is the only HLA source here.
df_map = pd.read_csv('data/mhc_mapping_demo.csv')

df = pd.DataFrame(columns=['MHC', 'MHC_alpha', 'MHC_beta', 'Epitope'])

# --- Chain dropdowns ---
# Split on the locus, not on the letter: every 'HLA-' name contains an A.
_n = df_map['HLA_Name'].astype(str)
_alpha = _n.str.match(r'HLA-D[PQR]A') | (_n.str.startswith('H2') & _n.str.endswith('A'))
_beta = _n.str.match(r'HLA-D[PQR]B') | (_n.str.startswith('H2') & _n.str.endswith('B'))
hla_list_a = sorted(_n[_alpha].unique())   # 27
hla_list_b = sorted(_n[_beta].unique())    # 89
assert len(hla_list_a) + len(hla_list_b) == len(df_map), "unclassified chain in mhc_mapping_demo.csv"

mhc_alpha_dropdown = widgets.Dropdown(options=hla_list_a, description="MHC alpha:", style={'description_width': 'initial'})
mhc_beta_dropdown = widgets.Dropdown(options=hla_list_b, description="MHC beta:", style={'description_width': 'initial'})

# --- Other widgets/functions ---
file_path_box = widgets.Text(
    value='data/dataset_demo.csv',
    placeholder='Enter CSV path (e.g., ./dataset.csv)',
    description='Load from:',
    style={'description_width': 'initial'}
)
export_path_box = widgets.Text(
    value='outputs/my_dataset.csv',
    placeholder='Enter CSV path to write (e.g., outputs/my_dataset.csv)',
    description='Export to:',
    style={'description_width': 'initial'}
)
load_button = widgets.Button(description="Load CSV")
add_button = widgets.Button(description="Add")
export_button = widgets.Button(description="Export CSV")
reset_button = widgets.Button(description="Reset", button_style='danger')
epitope_textbox = widgets.Text(
    value='', placeholder='Max 15 chars, uppercase', description='Epitope:', style={'description_width': 'initial'}
)
epitope_msg = widgets.HTML("Currently 0/15 characters entered")
output_area = widgets.Output()

def make_preferences_box():
    global num_workers, batch_size, plot_kde, use_compile, out_path, show_top_binders
    num_workers = widgets.IntSlider(
        value=8 if os.cpu_count() > 8 else os.cpu_count(),
        description="Num workers:",
        style={'description_width': 'initial'},
        min=0,
        max=os.cpu_count()
    )
    batch_size = widgets.IntSlider(
        value=min(512, len(df)),
        description="Batch size:",
        style={'description_width': 'initial'},
        min=1,
        max=max(1, len(df))
    )
    plot_kde = widgets.Checkbox(
        value=True,
        description="Plot KDE",
        style={'description_width': 'initial'}
    )
    show_top_binders = widgets.Dropdown(
        options=[('None', None), ('Top 5', 5), ('Top 10', 10)],
        value=5,
        description="Show top binders:",
        style={'description_width': 'initial'}
    )
    use_compile = widgets.Checkbox(
        value=False,
        description="Use torch compile",
        style={'description_width': 'initial'}
    )
    out_path = widgets.Text(
        value='outputs',
        placeholder='Enter output folder path (e.g., ./outputs)',
        description='Output folder:',
        style={'description_width': 'initial'}
    )
    return widgets.VBox([
        out_path, show_top_binders, plot_kde, num_workers, batch_size, use_compile
    ])

def refresh_ui():
    with output_area:
        clear_output()
        if df.empty:
            df_widget = widgets.HTML('<b>No data entered.</b>')
            ui = widgets.VBox([
                widgets.HBox([mhc_alpha_dropdown, mhc_beta_dropdown, epitope_textbox, add_button]),
                epitope_msg,
                widgets.HBox([file_path_box, load_button, reset_button]),
                widgets.HBox([export_path_box, export_button]),
                df_widget
            ])
        else:
            df_widget = widgets.HTML(df.tail(5).to_html(index=False))
            preferences_box = make_preferences_box()
            ui = widgets.VBox([
                widgets.HBox([mhc_alpha_dropdown, mhc_beta_dropdown, epitope_textbox, add_button]),
                epitope_msg,
                widgets.HBox([file_path_box, load_button, reset_button]),
                widgets.HBox([export_path_box, export_button]),
                df_widget,
                widgets.HTML('<hr>'),
                preferences_box
            ])
        display(ui)

def add_row(_):
    global df
    alpha, beta, epitope = mhc_alpha_dropdown.value, mhc_beta_dropdown.value, epitope_textbox.value
    if not (alpha and beta and epitope.strip()):
        with output_area:
            clear_output()
            display(widgets.HTML('<b style="color:red;">MHC alpha, MHC beta, and Epitope must all be entered.</b>'))
            refresh_ui()
    else:
        # beta_alpha, the key every shipped table uses (data/dataset/*, analysis/scoring/*).
        # The head has no positional encoding, so the order does not move the score; this is
        # about a widget-built frame being comparable to the corpus.
        mhc = beta + '_' + alpha
        df.loc[len(df)] = [mhc, alpha, beta, epitope]
        epitope_textbox.value = ""
        refresh_ui()

def export_csv(_):
    path = export_path_box.value
    with output_area:
        clear_output()
        # data/ holds the shipped demo assets; a stray export must not overwrite them.
        if os.path.abspath(path).startswith(os.path.abspath('data') + os.sep):
            display(widgets.HTML('<b style="color:red;">Refusing to write inside data/: pick another path.</b>'))
        else:
            os.makedirs(os.path.dirname(path) or '.', exist_ok=True)
            df.to_csv(path, index=False)
            display(widgets.HTML(f'<b>Saved: {path}</b>'))
        refresh_ui()

def load_csv(_):
    global df
    path = file_path_box.value
    with output_area:
        clear_output()
        if not os.path.isfile(path):
            display(widgets.HTML(f'<b style="color:red;">File not found: {path}</b>'))
        else:
            try:
                df = pd.read_csv(path)
                display(widgets.HTML(f'<b>Loaded file: {path}<br>Top 3 rows:</b>'))
                display(widgets.HTML(df.head(3).to_html(index=False)))
            except Exception as e:
                display(widgets.HTML(f'<b style="color:red;">Failed to read file: {str(e)}</b>'))
        refresh_ui()

def epitope_textbox_change(change):
    new_val = change['new'].upper()[:15]
    if new_val != epitope_textbox.value:
        epitope_textbox.value = new_val
        return
    epitope_msg.value = f"Currently {len(epitope_textbox.value)}/15 characters entered"

def reset_df(_):
    global df
    df = pd.DataFrame(columns=['MHC', 'MHC_alpha', 'MHC_beta', 'Epitope'])
    with output_area:
        clear_output()
        display(widgets.HTML('<b style="color:green;">DataFrame has been reset.</b>'))
        refresh_ui()

add_button.on_click(add_row)
export_button.on_click(export_csv)
load_button.on_click(load_csv)
reset_button.on_click(reset_df)
epitope_textbox.observe(epitope_textbox_change, names='value')

display(output_area)
refresh_ui()

In [ ]:
#@title 3. Run Prediction

if len(df) == 0:
    raise ValueError("Please add data before running the prediction.")

os.makedirs(out_path.value, exist_ok=True)
test_path = f'{out_path.value}/dataset.csv'
df.to_csv(test_path, index=False)

# config_path, chkp_path and esm_chkp_path all come from the Setup cell -- in particular the two
# checkpoint paths are the files that were actually downloaded, not a guess at where they landed.
config = load_config(
    config_path,
    chkp_path=chkp_path,
    esm_chkp_path=esm_chkp_path,
    num_workers=num_workers.value,
    batch_size=batch_size.value,
    use_compile=use_compile.value,
    test_path=test_path,
    plot=plot_kde.value,
    out_path=out_path.value,
)

df_out = run_inference(config)
cols = ['Score', 'Logits']
df_out[cols] = df_out[cols].apply(pd.to_numeric, errors='coerce')
df_out = df_out.nlargest(show_top_binders.value, 'Score') if show_top_binders.value is not None else df_out.sort_values('Score', ascending=False)
for col in cols:
    df_out[col] = df_out[col].map(lambda x: f"{x:.5f}")
display(df_out)

In [ ]:
#@title 4. Export Results
from google.colab import files

for _f in [f"{out_path.value}/prediction.csv"] + ([f"{out_path.value}/plot.png"] if plot_kde.value else []):
    if os.path.exists(_f):
        files.download(_f)
    else:
        print(f"not found, skipped: {_f}")